# 07: toorPIA可視化 — Hidden State構造の確認

**目的**: v1データ（200サンプル × 3072次元）をtoorPIAに投入し、3エージェント×2ラウンドの構造が見えるか確認する。

**判断基準**:
1. 3エージェントがマップ上で分離するか → しないなら「分離」アプローチ自体に疑問
2. Round 0→Round 1で移動が見えるか → 見えないならdebateの効果がhidden stateに反映されていない

**手順**:
- Map A: エージェント別分割（600サンプル × 1024次元）→ agent_id / round_id で色分け
- Map B: 全体（200サンプル × 3072次元）→ round_id で色分け
- Map C: PCA64次元圧縮版（600サンプル × 64次元）→ 有効次元のみで構造確認

In [1]:
# === Cell 1: セットアップ ===
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('thoughtcomm'):
        !git clone https://github.com/AUMEZAK/thoughtcomm.git
    %cd thoughtcomm
    !pip install -q toorpia
    from google.colab import drive
    drive.mount('/content/drive')

import torch
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from toorpia import toorPIA

TOORPIA_API_KEY = 'toorpia_8acfd4e501d10123bd18bf5398051394c37cb08b0546b4a0'
DRIVE_DIR = '/content/drive/MyDrive/thoughtcomm_checkpoints/'

print('Setup done.')

Cloning into 'thoughtcomm'...
remote: Enumerating objects: 395, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 395 (delta 16), reused 19 (delta 7), pack-reused 357 (from 1)
Receiving objects: 100% (395/395), 3.97 MiB | 10.56 MiB/s, done.
Resolving deltas: 100% (221/221), done.
/content/thoughtcomm
ERROR: Could not find a version that satisfies the requirement toorpia (from versions: none)
ERROR: No matching distribution found for toorpia
Mounted at /content/drive


ModuleNotFoundError: No module named 'toorpia'

In [ ]:
# === Cell 2: データ読み込み + メタデータ付与 ===
math_path = os.path.join(DRIVE_DIR, 'Qwen3-0.6B_math', 'hidden_states.pt')
gsm8k_path = os.path.join(DRIVE_DIR, 'Qwen3-0.6B_gsm8k', 'hidden_states.pt')

# mathデータを使用（gsm8kも後で比較可能）
ckpt = torch.load(math_path, map_location='cpu')
if 'H' in ckpt:
    H = ckpt['H'].float()
elif 'H_list' in ckpt:
    H = torch.stack(ckpt['H_list']).float()
meta = ckpt.get('metadata', [])

n_samples, n_h = H.shape
hidden_size = n_h // 3  # 1024

print(f'H shape: {H.shape}  ({n_samples} samples, {hidden_size} per agent × 3)')
print(f'metadata keys: {meta[0].keys() if meta else "empty"}')
print(f'Rounds: {sorted(set(m["round"] for m in meta))}')
print(f'Questions: {len(set(m["example_idx"] for m in meta))}')

In [ ]:
# === Cell 3: Map A — エージェント別分割（600 × 1024） ===
# 各200サンプルを3エージェントに分割 → 600サンプル × 1024次元
# メタデータ: agent_id, round_id, question_id

H_np = H.numpy()
rows = []
for idx in range(n_samples):
    m = meta[idx]
    for agent_id in range(3):
        h_agent = H_np[idx, agent_id * hidden_size : (agent_id + 1) * hidden_size]
        row = {f'dim_{i}': h_agent[i] for i in range(hidden_size)}
        row['agent_id'] = agent_id
        row['round_id'] = m['round']
        row['question_id'] = m['example_idx']
        rows.append(row)

df_agents = pd.DataFrame(rows)
print(f'df_agents shape: {df_agents.shape}')
print(f'  agent_id counts: {df_agents["agent_id"].value_counts().to_dict()}')
print(f'  round_id counts: {df_agents["round_id"].value_counts().to_dict()}')

# CSV保存
os.makedirs('results/toorpia', exist_ok=True)
csv_path_a = 'results/toorpia/map_a_agents_1024d.csv'
df_agents.to_csv(csv_path_a, index=False)
print(f'Saved: {csv_path_a}')

In [ ]:
# === Cell 4: Map A — toorPIA basemap投入 ===
client = toorPIA(api_key=TOORPIA_API_KEY)

print('Creating Map A: 600 samples × 1024 dims (agent-split)')
result_a = client.basemap_csvform(
    csv_path_a,
    drop_columns=['agent_id', 'round_id', 'question_id'],
    label='ThoughtComm math: agent-split 1024d',
    tag='thoughtcomm-hidden-state',
)

print(f'Map A created!')
print(f'  mapNo: {result_a["mapNo"]}')
print(f'  shareUrl: {result_a["shareUrl"]}')
print(f'  xyData shape: {result_a["xyData"].shape}')

In [ ]:
# === Cell 5: Map A — 結果の可視化（agent / round色分け） ===
import matplotlib.pyplot as plt

xy = result_a['xyData']  # (600, 2)
agents = df_agents['agent_id'].values
rounds = df_agents['round_id'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左: エージェント色分け
colors_agent = ['tab:blue', 'tab:orange', 'tab:green']
for a in range(3):
    mask = agents == a
    axes[0].scatter(xy[mask, 0], xy[mask, 1], c=colors_agent[a],
                    alpha=0.5, s=20, label=f'Agent {a}')
axes[0].set_title('Map A: Agent color')
axes[0].legend()
axes[0].set_xlabel('toorPIA x')
axes[0].set_ylabel('toorPIA y')

# 右: ラウンド色分け
colors_round = ['tab:red', 'tab:purple']
for r in sorted(set(rounds)):
    mask = rounds == r
    axes[1].scatter(xy[mask, 0], xy[mask, 1], c=colors_round[int(r)],
                    alpha=0.5, s=20, label=f'Round {int(r)}')
axes[1].set_title('Map A: Round color')
axes[1].legend()
axes[1].set_xlabel('toorPIA x')
axes[1].set_ylabel('toorPIA y')

plt.tight_layout()
plt.savefig('results/toorpia/map_a_scatter.png', dpi=150)
plt.show()

# エージェント×ラウンドの6グループ
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
markers = ['o', 's']  # round 0 = circle, round 1 = square
for a in range(3):
    for r in sorted(set(rounds)):
        mask = (agents == a) & (rounds == r)
        ax.scatter(xy[mask, 0], xy[mask, 1], c=colors_agent[a],
                   marker=markers[int(r)], alpha=0.5, s=25,
                   label=f'Agent {a} / Round {int(r)}')
ax.set_title('Map A: Agent × Round')
ax.legend()
ax.set_xlabel('toorPIA x')
ax.set_ylabel('toorPIA y')
plt.tight_layout()
plt.savefig('results/toorpia/map_a_agent_round.png', dpi=150)
plt.show()

In [ ]:
# === Cell 6: Map B — 全体（200 × 3072）===
# 3エージェント連結のまま投入 → ラウンド間の変化を見る

rows_b = []
for idx in range(n_samples):
    m = meta[idx]
    row = {f'dim_{i}': H_np[idx, i] for i in range(n_h)}
    row['round_id'] = m['round']
    row['question_id'] = m['example_idx']
    rows_b.append(row)

df_full = pd.DataFrame(rows_b)
csv_path_b = 'results/toorpia/map_b_full_3072d.csv'
df_full.to_csv(csv_path_b, index=False)
print(f'df_full shape: {df_full.shape}')

print('Creating Map B: 200 samples × 3072 dims (full concatenated)')
result_b = client.basemap_csvform(
    csv_path_b,
    drop_columns=['round_id', 'question_id'],
    label='ThoughtComm math: full 3072d',
    tag='thoughtcomm-hidden-state',
)

print(f'Map B created!')
print(f'  mapNo: {result_b["mapNo"]}')
print(f'  shareUrl: {result_b["shareUrl"]}')

# 可視化
xy_b = result_b['xyData']
rounds_b = df_full['round_id'].values
questions_b = df_full['question_id'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ラウンド色分け
for r in sorted(set(rounds_b)):
    mask = rounds_b == r
    axes[0].scatter(xy_b[mask, 0], xy_b[mask, 1], alpha=0.5, s=30,
                    label=f'Round {int(r)}')
axes[0].set_title('Map B: Round color (200 × 3072d)')
axes[0].legend()

# 同一問題のRound 0→Round 1を矢印で結ぶ
unique_q = sorted(set(questions_b))
for q in unique_q:
    r0_mask = (questions_b == q) & (rounds_b == 0)
    r1_mask = (questions_b == q) & (rounds_b == 1)
    if r0_mask.sum() == 1 and r1_mask.sum() == 1:
        x0, y0 = xy_b[r0_mask][0]
        x1, y1 = xy_b[r1_mask][0]
        axes[1].annotate('', xy=(x1, y1), xytext=(x0, y0),
                         arrowprops=dict(arrowstyle='->', color='gray', alpha=0.3))
        axes[1].scatter(x0, y0, c='tab:red', s=15, alpha=0.6)
        axes[1].scatter(x1, y1, c='tab:purple', s=15, alpha=0.6)

axes[1].set_title('Map B: Round 0→1 movement per question')
# Manual legend
axes[1].scatter([], [], c='tab:red', s=30, label='Round 0')
axes[1].scatter([], [], c='tab:purple', s=30, label='Round 1')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/toorpia/map_b_scatter.png', dpi=150)
plt.show()

# 移動距離の統計
dists = []
for q in unique_q:
    r0_mask = (questions_b == q) & (rounds_b == 0)
    r1_mask = (questions_b == q) & (rounds_b == 1)
    if r0_mask.sum() == 1 and r1_mask.sum() == 1:
        d = np.linalg.norm(xy_b[r0_mask][0] - xy_b[r1_mask][0])
        dists.append(d)
dists = np.array(dists)
print(f'\nRound 0→1 移動距離 (toorPIA座標): mean={dists.mean():.3f}, std={dists.std():.3f}, max={dists.max():.3f}')

In [ ]:
# === Cell 7: Map C — PCA64次元圧縮版（600 × 64） ===
# 有効次元95% = 64次元に圧縮してからtoorPIA投入

# エージェント分割データ（600 × 1024）にPCA適用
dim_cols = [f'dim_{i}' for i in range(hidden_size)]
X = df_agents[dim_cols].values  # (600, 1024)
X_centered = X - X.mean(axis=0)

pca = PCA(n_components=64)
X_pca = pca.fit_transform(X_centered)
print(f'PCA: {hidden_size}d → 64d, explained variance = {pca.explained_variance_ratio_.sum():.1%}')

# CSV作成
df_pca = pd.DataFrame(X_pca, columns=[f'pc_{i}' for i in range(64)])
df_pca['agent_id'] = df_agents['agent_id'].values
df_pca['round_id'] = df_agents['round_id'].values
df_pca['question_id'] = df_agents['question_id'].values

csv_path_c = 'results/toorpia/map_c_pca64d.csv'
df_pca.to_csv(csv_path_c, index=False)

print('Creating Map C: 600 samples × 64 dims (PCA-reduced agent-split)')
result_c = client.basemap_csvform(
    csv_path_c,
    drop_columns=['agent_id', 'round_id', 'question_id'],
    label='ThoughtComm math: PCA64d agent-split',
    tag='thoughtcomm-hidden-state',
)

print(f'Map C created!')
print(f'  mapNo: {result_c["mapNo"]}')
print(f'  shareUrl: {result_c["shareUrl"]}')

In [ ]:
# === Cell 8: Map C — 結果の可視化 ===
xy_c = result_c['xyData']
agents_c = df_pca['agent_id'].values
rounds_c = df_pca['round_id'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# エージェント色分け
for a in range(3):
    mask = agents_c == a
    axes[0].scatter(xy_c[mask, 0], xy_c[mask, 1], c=colors_agent[a],
                    alpha=0.5, s=20, label=f'Agent {a}')
axes[0].set_title('Map C (PCA64d): Agent color')
axes[0].legend()

# Agent × Round
for a in range(3):
    for r in sorted(set(rounds_c)):
        mask = (agents_c == a) & (rounds_c == r)
        axes[1].scatter(xy_c[mask, 0], xy_c[mask, 1], c=colors_agent[a],
                        marker=markers[int(r)], alpha=0.5, s=25,
                        label=f'Agent {a} / Round {int(r)}')
axes[1].set_title('Map C (PCA64d): Agent × Round')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/toorpia/map_c_scatter.png', dpi=150)
plt.show()

In [ ]:
# === Cell 9: gsm8kデータでも同様に確認 ===
if os.path.exists(gsm8k_path):
    ckpt_g = torch.load(gsm8k_path, map_location='cpu')
    if 'H' in ckpt_g:
        H_g = ckpt_g['H'].float()
    elif 'H_list' in ckpt_g:
        H_g = torch.stack(ckpt_g['H_list']).float()
    meta_g = ckpt_g.get('metadata', [])

    H_g_np = H_g.numpy()
    rows_g = []
    for idx in range(len(meta_g)):
        m = meta_g[idx]
        for agent_id in range(3):
            h_agent = H_g_np[idx, agent_id * hidden_size : (agent_id + 1) * hidden_size]
            row = {f'dim_{i}': h_agent[i] for i in range(hidden_size)}
            row['agent_id'] = agent_id
            row['round_id'] = m['round']
            row['question_id'] = m['example_idx']
            rows_g.append(row)

    df_gsm8k = pd.DataFrame(rows_g)
    csv_path_g = 'results/toorpia/map_gsm8k_agents_1024d.csv'
    df_gsm8k.to_csv(csv_path_g, index=False)

    # addplot: mathのbasemapにgsm8kを重ねる
    print(f'Addplot: gsm8k onto Map A (math basemap, mapNo={result_a["mapNo"]})')
    result_g = client.addplot_csvform(
        csv_path_g,
        mapNo=result_a['mapNo'],
    )
    print(f'  abnormalityScore: {result_g.get("abnormalityScore", "N/A")}')
    print(f'  abnormalityStatus: {result_g.get("abnormalityStatus", "N/A")}')
    print(f'  shareUrl: {result_g.get("shareUrl", "N/A")}')
else:
    print('gsm8k data not found, skipping')

In [ ]:
# === Cell 10: 結果サマリー + 判断 ===
import json

print('=' * 70)
print('toorPIA Visualization Summary')
print('=' * 70)

print(f'\nMap A (agent-split, 600 × 1024d):')
print(f'  mapNo: {result_a["mapNo"]}')
print(f'  shareUrl: {result_a["shareUrl"]}')

print(f'\nMap B (full, 200 × 3072d):')
print(f'  mapNo: {result_b["mapNo"]}')
print(f'  shareUrl: {result_b["shareUrl"]}')
print(f'  Round 0→1 移動距離: mean={dists.mean():.3f}')

print(f'\nMap C (PCA64d agent-split, 600 × 64d):')
print(f'  mapNo: {result_c["mapNo"]}')
print(f'  shareUrl: {result_c["shareUrl"]}')

print(f'\n=== 判断基準 ===')
print('Q1: 3エージェントがマップ上で分離するか？')
print('  → Map A / Map C の散布図でagent色が分離していれば YES')
print('Q2: Round 0→Round 1で移動が見えるか？')
print('  → Map B の矢印図で系統的な移動があれば YES')
print('\n上のMap URLをブラウザで開いて、toorPIA Inspector上でも確認してください。')

# 結果保存
summary = {
    'map_a': {'mapNo': result_a['mapNo'], 'shareUrl': result_a['shareUrl'],
              'desc': 'agent-split 600x1024d'},
    'map_b': {'mapNo': result_b['mapNo'], 'shareUrl': result_b['shareUrl'],
              'desc': 'full 200x3072d', 'round_move_dist_mean': float(dists.mean())},
    'map_c': {'mapNo': result_c['mapNo'], 'shareUrl': result_c['shareUrl'],
              'desc': 'PCA64d agent-split 600x64d'},
}
with open('results/toorpia/07_toorpia_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSaved: results/toorpia/07_toorpia_summary.json')